In [ ]:
import (
	"fmt"
	"math/rand"
	"time"
	"os"
)

In [1]:
%%
fmt.Println("hello world")

hello world


In [2]:
func sayHello() {
	fmt.Println("hello world")
}

In [3]:
%%
 // go sayHello()
 // Anonymous goroutines
 go func() {
	fmt.Println("Hello!")
 }()

In [4]:
%%
// Through named functions
var wg sync.WaitGroup
sayHello := func() {
	defer wg.Done()
	fmt.Println("Hello!")
}
wg.Add(1)
go sayHello()
wg.Wait()

Hello!


In [5]:
%%
// Through closures
var wg sync.WaitGroup
salutation := "hello"
wg.Add(1)
go func() {
	defer wg.Done()
	salutation = "welcome"
}()
wg.Wait()
fmt.Println(salutation)

welcome


In [6]:
%%
var wg sync.WaitGroup
for _, salutation := range []string{"hello", "greetings", "good day"} {
	wg.Add(1)
	go func() {
		defer wg.Done()
		fmt.Println(salutation)
	}()
}
wg.Wait()

good day
hello
greetings


In [7]:
%% 
memConsumed := func() uint64 {
	runtime.GC()
	var s runtime.MemStats
	runtime.ReadMemStats(&s)
	return s.Sys
}
var c <-chan interface{}
var wg sync.WaitGroup
noop := func() { wg.Done(); <-c }
const numGoroutines = 1e4
wg.Add(numGoroutines)
before := memConsumed()
for i := numGoroutines; i > 0; i-- {
	go noop()
}
wg.Wait()
after := memConsumed()
fmt.Printf("%.3fkb", float64(after-before)/numGoroutines/1000)

2.549kb

In [8]:
!taskset -c 0 perf bench sched pipe -T

# Running 'sched/pipe' benchmark:


# Executed 1000000 pipe operations between two threads

     Total time: 11.407 [sec]

      11.407322 usecs/op
          87662 ops/sec


In [9]:
%%

file, err := os.Create("context-switching_test.go")
if err != nil {
	fmt.Println(err)
	return
}

defer file.Close()
content := `package main

import (
	"sync"
	"testing"
)

func BenchmarkContextSwitch(b *testing.B) {
	var wg sync.WaitGroup
	begin := make(chan struct{})
	c := make(chan struct{})
	var token struct{}

	sender := func() {
		defer wg.Done()
		<-begin
		for i := 0; i < b.N; i++ {
			c <- token
		}
	}

	receiver := func() {
		defer wg.Done()
		<-begin
		for i := 0; i < b.N; i++ {
			<-c
		}
	}

	wg.Add(2)
	go sender()
	go receiver()
	b.StartTimer()
	close(begin)
	wg.Wait()
}`
_, err = file.WriteString(content)
if err != nil {
	fmt.Println(err)
	return
}
fmt.Println("file created")


file created


In [10]:
!go test -bench=. -cpu=1 context-switching_test.go

goos: linux
goarch: arm64
BenchmarkContextSwitch 	 1745696	       695.9 ns/op
PASS
ok  	command-line-arguments	1.917s


In [11]:
%%

var wg sync.WaitGroup
wg.Add(1)
go func() {
	defer wg.Done()
	fmt.Println("1st goroutine sleeping...")
	time.Sleep(1)
}()
wg.Add(1)
go func() {
	defer wg.Done()
	fmt.Println("2nd goroutine sleeping...")
	time.Sleep(1)
}()
wg.Wait()
fmt.Println("All goroutines complete.")

1st goroutine sleeping...
2nd goroutine sleeping...
All goroutines complete.


In [12]:
%%

var wg sync.WaitGroup
helloworld := func(wg *sync.WaitGroup, id int) {
	defer wg.Done()
	fmt.Printf("Hello from %v!\n", id)
}

const numGreeters = 5
wg.Add(numGreeters)
for i := 0; i < numGreeters; i++ {
	go helloworld(&wg, i)	
}
wg.Wait()

Hello from 1!
Hello from 2!
Hello from 4!
Hello from 3!
Hello from 0!


In [13]:
%%
// Mutex

var data int
var mutex sync.Mutex

increment := func() {
	mutex.Lock()
	defer mutex.Unlock()
	data++
	fmt.Printf("Incrementing %v\n", data)
}

decrement := func() {
	mutex.Lock()
	defer mutex.Unlock()
	data--
	fmt.Printf("Decrementing %v\n", data)
}

var arithmetic sync.WaitGroup
for i := 0; i <= 5; i++ {
	arithmetic.Add(1)
	go func() {
		defer arithmetic.Done()
		increment()
	}()
}

for i := 0; i <= 5; i++ {
	arithmetic.Add(1)
	go func() {
		defer arithmetic.Done()
		decrement()
	}()
}

arithmetic.Wait()
fmt.Println("Arithmetic complete.")

Incrementing 1
Decrementing 0
Incrementing 1
Incrementing 2
Incrementing 3
Decrementing 2
Decrementing 1
Decrementing 0
Decrementing -1
Incrementing 0
Incrementing 1
Decrementing 0
Arithmetic complete.


In [14]:
%%
producer := func(wg *sync.WaitGroup, l sync.Locker) {
	defer wg.Done()
	for i := 5; i > 0; i-- {
		l.Lock()
		l.Unlock()
		time.Sleep(1)
	}
}

consumer := func(wg *sync.WaitGroup, l sync.Locker) {
	defer wg.Done()
		l.Lock()
		defer l.Unlock()
	}

test := func(count int, mutex, rwMutex sync.Locker) time.Duration {
	var wg sync.WaitGroup
	wg.Add(count + 1)
	begin := time.Now()
	go producer(&wg, mutex)
	for i := count; i > 0; i-- {
		go consumer(&wg, rwMutex)
	}
	wg.Wait()
	return time.Since(begin)
}

tw := tabwriter.NewWriter(os.Stdout, 0, 1, 2, ' ', 0)
defer tw.Flush()

var m sync.RWMutex
fmt.Fprintf(tw, "Readers\tMutex\tRWMutex\n")
for i := 0; i < 20; i++ {
	count := 1 << uint(i)
	fmt.Fprintf(tw, 
		"%d\t%v\t%v\n", 
		count, 
		test(count, &m, m.RLocker()), 
		test(count, &m, &m))
}

Readers  Mutex         RWMutex
1        72.481µs      42.315µs
2        23.666µs      12.703µs
4        87.239µs      42.351µs
8        27.11µs       52.611µs
16       90.888µs      44.814µs
32       110.795µs     129.294µs
64       210.182µs     77.61µs
128      139.739µs     98.813µs
256      320.829µs     210.923µs
512      382.124µs     370.161µs
1024     587.714µs     586.843µs
2048     1.079855ms    1.356239ms
4096     2.529056ms    2.153247ms
8192     4.57923ms     4.453472ms
16384    9.366605ms    14.694176ms
32768    19.6539ms     24.332184ms
65536    38.285759ms   41.690635ms
131072   75.350332ms   84.884711ms
262144   161.421211ms  172.348423ms
524288   340.468592ms  371.030835ms


In [15]:
c := sync.NewCond(&sync.Mutex{})
c.L.Lock()
for conditionTrue() == false {
	c.Wait()
}
c.L.Unlock()

Cell[15]: Line 1 
 /tmp/gonb_e2983c85/main.go:3:1: expected declaration, found c
 
package main

 c := sync.NewCond(&sync.Mutex{})
 c.L.Lock()
for conditionTrue() == false {

ERROR: parsing go files in TempDir "/tmp/gonb_e2983c85": /tmp/gonb_e2983c85/main.go:3:1: expected declaration, found c

In [ ]:
%%

c := sync.NewCond(&sync.Mutex{})
queue := make([]interface{}, 0, 10)
removeFromQueue := func(delay time.Duration) {
	time.Sleep(delay)
	c.L.Lock()
	queue = queue[1:]
	fmt.Println("Removed from queue")
	c.L.Unlock()
	c.Signal()
}

for i := 0; i < 10; i++ {
	c.L.Lock()
	for len(queue) == 2 {
		c.Wait()
	}
	fmt.Println("Adding to queue")
	queue = append(queue, struct{}{})
	go removeFromQueue(1 * time.Second)
	c.L.Unlock()
}

Adding to queue
Adding to queue
Removed from queue
Removed from queue
Adding to queue
Adding to queue
Removed from queue
Adding to queue
Removed from queue
Adding to queue
Removed from queue
Adding to queue
Removed from queue
Adding to queue
Removed from queue
Adding to queue
Removed from queue
Adding to queue


In [ ]:

type Button struct {
	Clicked *sync.Cond
}

%%

subscribe := func(c *sync.Cond, fn func()) {
	var swg sync.WaitGroup
	swg.Add(1)
	go func() {
		swg.Done()
		c.L.Lock()
		defer c.L.Unlock()
		c.Wait()
		fn()
	}()
	swg.Wait()
}

button := Button{Clicked: sync.NewCond(&sync.Mutex{})}

var cwg sync.WaitGroup
cwg.Add(3)
subscribe(button.Clicked, func() {
	fmt.Println("Maximizing window.")
	cwg.Done()
})
subscribe(button.Clicked, func() {
	fmt.Println("Displaying annoying dialog box!")
	cwg.Done()
})
subscribe(button.Clicked, func() {
	fmt.Println("Mouse clicked.")
	cwg.Done()
})
button.Clicked.Broadcast()
cwg.Wait()


Mouse clicked.
Maximizing window.
Displaying annoying dialog box!


In [ ]:
%%
var count int
increment := func() {
	count++
	fmt.Printf("Incrementing: %d\n", count)
}

var once sync.Once
var increments sync.WaitGroup
increments.Add(100)
for i := 0; i < 100; i++ {
	go func() {
		defer increments.Done()
		once.Do(increment)
	}()
}
increments.Wait()
fmt.Printf("Count is %d\n", count)

Incrementing: 1
Count is 1


In [ ]:
%%
myPool := &sync.Pool{
	New: func() interface{} {
		fmt.Println("Creating new instance.")
		return struct{}{}
	},
}
myPool.Get()
instance := myPool.Get()
myPool.Put(instance)
myPool.Get()

Creating new instance.
Creating new instance.


In [ ]:
%%
var numCalcsCreated int
calcPool := &sync.Pool{
	New: func() interface{} {
		numCalcsCreated++
		mem := make([]byte, 1024)
		return &mem
	},
}
calcPool.Put(calcPool.New())
calcPool.Put(calcPool.New())
calcPool.Put(calcPool.New())
calcPool.Put(calcPool.New())

const numWorkers = 1024*1024
var wg sync.WaitGroup
wg.Add(numWorkers)
for i := numWorkers; i > 0; i-- {
	go func() {
		defer wg.Done()
		mem := calcPool.Get().(*[]byte)
		defer calcPool.Put(mem)
	}()
}
wg.Wait()
fmt.Printf("%d calculators were created.", numCalcsCreated)

5 calculators were created.

In [ ]:
!cd sync-pool && go test -benchtime=10s -bench=.


goos: linux
goarch: arm64
pkg: rachitmishra.com/go/notebooks/sync-pool
BenchmarkNetworkRequest-4   	   19803	   1036862 ns/op
PASS
ok  	rachitmishra.com/go/notebooks/sync-pool	35.176s


In [ ]:
// Channels
%%
var dataStream chan interface{}
dataStream = make(chan interface{})

fatal error: all goroutines are asleep - deadlock!

goroutine 1 [chan receive]:
main.main()
	 [[ Cell [18] Line 41 ]] /tmp/gonb_f5e8d9f7/main.go:49 +0x94
exit status 2


In [ ]:
%%
var readOnlyDataStream <-chan interface{}
readOnlyDataStream = make(<-chan interface{})

var writeOnlyDataStream chan<- interface{}
writeOnlyDataStream = make(chan<- interface{})

In [ ]:
%%
var readStream <-chan interface{}
var writeStream chan<- interface{}

readWriteStream := make(chan interface{})

readStream = readWriteStream
writeStream = readWriteStream

In [ ]:
%%

intStream := make(chan int)

stringStream := make(chan string)
go func() {
	stringStream <- "Hello channels!"
}()
salutation, ok := <-stringStream
fmt.Printf("(%v): %v", ok, salutation)

In [ ]:
%%
writeStream := make(chan<- interface{})
readStream := make(<-chan interface{})
<-writeStream
readStream <- struct{}{}

In [ ]:
%%
stringStream := make(chan string)
go func() {
	return // nothing to send
	stringStream <- "Hello channels!"
}()
salutation, ok := <-stringStream
fmt.Printf("(%v): %v", ok, salutation)

fatal error: all goroutines are asleep - deadlock!

goroutine 1 [chan receive]:
main.main()
	 [[ Cell [21] Line 7 ]] /tmp/gonb_f5e8d9f7/main.go:16 +0x90
exit status 2


In [ ]:
%%
valueStream := make(chan interface{})
close(valueStream)

intStream := make(chan int)
close(intStream)
integer, ok := <-intStream
fmt.Printf("(%v): %v", ok, integer)

(false): 0

In [ ]:
%%
intStream := make(chan int)
go func() {
	defer close(intStream)
	for i := 1; i <= 5; i++ {
		intStream <- i
	}
}()
for integer := range intStream {
	fmt.Printf("%v ", integer)
}


1 2 3 4 5 

In [ ]:
// Unblocking multiple goroutines
%%
begin := make(chan interface{})
var wg sync.WaitGroup
for i := 0; i < 5; i++ {
	wg.Add(1)
	go func() {
		defer wg.Done()
		fmt.Printf("%v has begun\n", i)
		<-begin
		fmt.Printf("%v has ended\n", i)
	}()
}
fmt.Println("Unblocking goroutines...")
close(begin)
wg.Wait()

0 has begun
Unblocking goroutines...
0 has ended
1 has begun
1 has ended
4 has begun
4 has ended
3 has begun
3 has ended
2 has begun
2 has ended


In [ ]:
// Buffered channels
%%
stringStream := make(chan string, 3)
stringStream <- "Hello"
stringStream <- "Buffered"
stringStream <- "Channel"
close(stringStream)

for s := range stringStream {
	fmt.Println(s)
}

fatal error: all goroutines are asleep - deadlock!

goroutine 1 [chan send]:
main.main()
	 [[ Cell [26] Line 7 ]] /tmp/gonb_f5e8d9f7/main.go:15 +0xb0
exit status 2


In [ ]:
%%
chanOwner := func() <-chan int {
	resultStream := make(chan int, 5)
	go func() {
		defer close(resultStream)
		for i := 0; i <= 5; i++ {
			resultStream <- i
		}
	}()
	return resultStream
}

resultStream := chanOwner()
for result := range resultStream {
	fmt.Printf("Received: %d\n", result)
}
fmt.Println("Done receiving!")

Received: 0
Received: 1
Received: 2
Received: 3
Received: 4
Received: 5
Done receiving!


In [ ]:
'''
Deadlocks:
Writing to nil channel blocks -> fatal error: all goroutines are asleep - deadlock!
Reading from a nil channel blocks -> fatal error: all goroutines are asleep - deadlock!

Panics:
Writing to a closed channel panics -> panic: send on nil channel
Closing a nil channel panics -> panic: close of nil channel
Closing a channel twice panics -> panic: close of closed channel
'''

In [ ]:
%%
var c1, c2 <-chan interface{}
var c3 chan<- interface{}

select {
case <-c1:
	fmt.Println("Received from c1")
case <-c2:
	fmt.Println("Received from c2")
case c3 <- struct{}{}:
	fmt.Println("Sent to c3")
default:
	fmt.Println("No communication")
}

No communication


In [ ]:
%%
start := time.Now()
c := make(chan interface{})
go func() {
	time.Sleep(5 * time.Second)
	close(c)
}()
fmt.Println("Blocking on read...")
select {
case <-c:
	fmt.Printf("Unblocked %v later.\n", time.Since(start))
}

Blocking on read...
Unblocked 5.004240786s later.


In [ ]:
%%
c1 := make(chan interface{}); close(c1)
c2 := make(chan interface{}); close(c2)

var c1Count, c2Count int
for i := 1000; i >= 0; i-- {
	select {
	case <-c1:
		c1Count++
	case <-c2:
		c2Count++
	}
}
fmt.Printf("c1Count: %d\nc2Count: %d\n", c1Count, c2Count)


c1Count: 488
c2Count: 513


In [ ]:
%%
var c<- chan interface{}
select {
case <-c:
	fmt.Println("This will never happen.")
case <-time.After(1 * time.Second):
	fmt.Println("Timed out.")
}

Timed out.


In [ ]:
%%
done := make(chan interface{})
go func() {
	time.Sleep(5 * time.Second)
	close(done)
}()

workCounter := 0
loop:
for {
	select {
	case <-done:
		break loop
	default:
	}
	workCounter++
	time.Sleep(1 * time.Second)
}
fmt.Printf("Achieved %v cycles of work before signalled to stop.\n", workCounter)


Achieved 6 cycles of work before signalled to stop.


In [ ]:
%%
data := make([]int, 4)
loopData := func(handleData chan<- int) {
	defer close(handleData)
	for _, datum := range data {
		handleData <- datum
	}
}
handleData := make(chan int)
go loopData(handleData)

for num := range handleData {
	fmt.Println(num)
}

0
0
0
0


In [ ]:
printData := func(wg *sync.WaitGroup, data []byte) {
	defer wg.Done()
	var buffer bytes.Buffer
	for _, datum := range data {
		fmt.Fprintf(&buffer, "%d\n", datum)
	}
	fmt.Println(buffer.String())
}

var wg sync.WaitGroup
wg.Add(2)
data := []byte("golang")
go printData(&wg, data[:3])
go printData(&wg, data[3:])

wg.Wait()

In [ ]:
for {
	select {
	case <-done:
		break
	default:
	}
}

In [ ]:
%%
doWork := func(strings <-chan string) <- chan interface{} {
	completed := make(chan interface{})
	go func() {
		defer fmt.Println("doWork exited.")
		defer close(completed)
		for s := range strings {
			fmt.Println(s)
		}
	}()
	return completed
}
doWork(nil)
fmt.Println("Done.")

Done.


In [ ]:
%%
doWork := func(done <-chan interface{}, strings <-chan string) <-chan interface{} {
	terminated := make(chan interface{})
	go func() {
		defer fmt.Println("doWork exited.")
		defer close(terminated)
		for {
			select {
			case s := <-strings:
				fmt.Println(s)
			case <-done:
				return
			}
		}
	}()
	return terminated
}
done := make(chan interface{})
terminated := doWork(done, nil)
go func() {
	time.Sleep(1 * time.Second)
	fmt.Println("Canceling doWork goroutine...")
	close(done)
}()
<-terminated

Canceling doWork goroutine...
doWork exited.


In [ ]:
%%
newRandStream := func(
	done <-chan interface{},
) <-chan int {
	randStream := make(chan int)
	go func() {
		defer fmt.Println("newRandStream closure exited.")
		defer close(randStream)
		for {
		select {
			case randStream <- rand.Int():
			case <-done:
				return
			}
		}
	}()
	return randStream
}

fmt.Println("3 random ints:")
done := make(chan interface{})
randStream := newRandStream(done)
for i:=1; i<=3; i++ {
	fmt.Println(<-randStream)
}
close(done)
time.Sleep(1 * time.Second)

3 random ints:
239499402024370973
5259559963823173966
3706307315798619227
newRandStream closure exited.


In [ ]:
func or(channels ...<-chan interface{}) <-chan interface{} {
	switch len(channels) {
	case 0:
		return nil
	case 1:
		return channels[0]
	}
	orDone := make(chan interface{})
	go func() {
		defer fmt.Println("or - closure exited.")
		defer close(orDone)
		switch len(channels) {
		case 2:
			select {
			case <-channels[0]:
			case <-channels[1]:
			}
		default:
			select {
			case <-channels[0]:
			case <-channels[1]:
			case <-channels[2]:
			case <-or(append(channels[3:], orDone)...):
			}
		}
	}()
	return orDone
}

In [ ]:
%%
sig := func(after time.Duration) <-chan interface{} {
	c := make(chan interface{})
	go func() {
		defer close(c)
		time.Sleep(after)
	}()
	return c
}
start := time.Now()
<-or(
	sig(2*time.Hour),
	sig(5*time.Minute),
	sig(1*time.Second),
	sig(1*time.Hour),
	sig(1*time.Minute),
)
fmt.Printf("done after %v", time.Since(start))

or - closure exited.
or - closure exited.
done after 1.000283722s

In [ ]:
%%
checkStatus := func(
	done <-chan interface{},
	urls ...string,
) <-chan *http.Response {
	responses := make(chan *http.Response)
	go func() {
		defer close(responses)
		for _, url := range urls {
			resp, err := http.Get(url)
			if err != nil {
				fmt.Printf("Error: %v\n", err)
				return
			}
			select {
			case <-done:
				return
			case responses <- resp:
			}
		}
	}()
	return responses
}
done := make(chan interface{})
defer close(done)
urls := []string{"https://www.google.com", "https://www.bing.com", "https://badhost"}
for response := range checkStatus(done, urls...) {
	fmt.Printf("Response: %v\n", response.Status)
}

Response: 200 OK
Response: 200 OK
Error: Get "https://badhost": dial tcp: lookup badhost on 100.100.100.100:53: no such host


In [ ]:
type Result struct {
	Error error
	Response *http.Response
}

%%
checkStatusWithResult := func(
	done <-chan interface{},
	urls ...string,
) <-chan Result {
	results := make(chan Result)
	go func() {
		defer close(results)
		for _, url := range urls {
			var result Result
			resp, err := http.Get(url)
			result = Result{Error: err, Response: resp}
			select {
			case <-done:
				return
			case results <- result:
			}
		}
	}()
	return results
}
done := make(chan interface{})
defer close(done)

urls := []string{"https://www.google.com", "https://badhost"}

for result := range checkStatusWithResult(done, urls...) {
	if result.Error != nil {
		fmt.Printf("Error: %v\n", result.Error)
		continue
	}
	fmt.Printf("Response: %v\n", result.Response.Status)
}

Response: 200 OK
Error: Get "https://badhost": dial tcp: lookup badhost on 100.100.100.100:53: no such host


In [ ]:
// Pipelines with channels
%%
generator := func(done <-chan interface{}, integers ...int) <-chan int {
	intStream := make(chan int)
	go func() {
		defer close(intStream)
		for _, i := range integers {
			select {
			case <-done:
				return
			case intStream <- i:
			}
		}
	}()
	return intStream
}
multiply := func(
	done <-chan interface{},
	intStream <-chan int,
	multiplier int,
) <-chan int {
	multipliedStream := make(chan int)
	go func() {
		defer close(multipliedStream)
		for i := range intStream {
			select {
			case <-done:
				return
			case multipliedStream <- i * multiplier:
			}
		}
	}()
	return multipliedStream
}
add := func(
	done <-chan interface{},
	intStream <-chan int,
	additive int,
) <-chan int {
	addedStream := make(chan int)
	go func() {
		defer close(addedStream)
		for i := range intStream {
			select {
			case <-done:
				return
			case addedStream <- i + additive:
			}
		}
	}()
	return addedStream
}

done := make(chan interface{})
defer close(done)

intStream := generator(done, 1, 2, 3, 4)
pipeline := multiply(done, add(done, multiply(done, intStream, 2), 1), 2)

for v := range pipeline {
	fmt.Println(v)
}


6
10
14
18


In [ ]:
func repeat(
	done <-chan interface{},
	values ...interface{},
) <-chan interface{} {
	valueStream := make(chan interface{})
	go func() {
		defer close(valueStream)
		for {
			for _, v := range values {
				select {
				case <-done:
					return
				case valueStream <- v:
				}
			}
		}
	}()
	return valueStream
}

In [ ]:
func take(
	done <-chan interface{},
	valueStream <-chan interface{},
	num int,
) <-chan interface{} {
	takeStream := make(chan interface{})
	go func() {
		defer close(takeStream)
		for i := num; i > 0 || num == -1; {
			if i != -1 {
				i--
			}
			select {
			case <-done:
				return
			case takeStream <- <-valueStream:
			}
		}
	}()
	return takeStream
}


In [ ]:
%%
done := make(chan interface{})
defer close(done)

for num := range take(done, repeat(done, 1), 10) {
	fmt.Printf("%v ", num)
}



1 1 1 1 1 1 1 1 1 1 

In [ ]:
func repeatFn(
	done <-chan interface{},
	fn func() interface{},
) (<-chan interface{} ){
	valueStream := make(chan interface{})
	go func() {
		defer close(valueStream)
		for {
			select {
			case <-done:
				return
			case valueStream <- fn():
			}
		}
	}()
	return valueStream
}

%%
done := make(chan interface{})
defer close(done)

rand := func() interface{} { return rand.Int() }

for num := range take(done, repeatFn(done, rand), 10) {
	fmt.Println(num)
}

In [ ]:
%%
fmt.Println("%d", runtime.NumCPU())

%d 4


In [ ]:
fanIn := func(
	done <-chan interface{},
	channels ...<-chan interface{},
) <-chan interface{} {
	var wg sync.WaitGroup
	multiplexedStream := make(chan interface{})
	multiplex := func(c <-chan interface{}) {
		defer wg.Done()
		for i := range c {
			select {
			case <-done:
				return
			case multiplexedStream <- i:
			}
		}
	}
	wg.Add(len(channels))
	for _, c := range channels {
		go multiplex(c)
	}
	go func() {
		wg.Wait()
		close(multiplexedStream)
	}()
	return multiplexedStream
}

In [ ]:
done := make(chan interface{})
defer close(done)

start := time.Now()

rand := func() interface{} { return rand.Intn(50000000) }

randIntStream := toInt(done, repeatFn(done, rand))
numFinders := runtime.NumCPU()
fmt.Printf("Spinning up %d prime finders.\n", numFinders)
finders := make([]<-chan interface{}, numFinders)
fmt.Println("Primes:")
for i := 0; i < numFinders; i++ {
	finders[i] = primeFinder(done, randIntStream)
}
for prime := range take(done, fanIn(done, finders...), 10) {
	fmt.Printf("\t%d\n", prime)
}
fmt.Printf("Search took: %v", time.Since(start))

In [ ]:
func orDone(done, c<- chan interface{}) <- chan interface{} {
	valStream := make(chan interface{})
	go func() {
		defer close(valStream)
		for {
			select {
			case <-done:
				return
			case v, ok := <-c:
				if ok == false {
					return
				}
				select {
				case valStream <- v:
				case <-done:
				}
			}
		}
	}()
	return valStream
}

In [ ]:
func tee(
	done <- chan interface{},
	in <- chan interface{},
) (<- chan interface{}, <- chan interface{}) {
	out1 := make(chan interface{})
	out2 := make(chan interface{})

	go func() {
		defer close(out1)
		defer close(out2)
		for val := range orDone(done, in) {
			var out1, out2 = out1, out2
			for i := 0; i < 2; i++ {
				select {
				case <-done:
				case out1 <- val:
					out1 = nil
				case out2 <- val:
					out2 = nil
				}
			}
		}
	}()
	return out1, out2
}


In [ ]:
%%
done := make(chan interface{})
defer close(done)

repeat := repeat(done, 1, 2)
take := take(done, repeat, 2)
out1, _ := tee(done, take)

for val1 := range out1 {
	fmt.Printf("out1: %v\n", val1)
}

In [ ]:
func bridge(
	done <- chan interface{},
	chanStream <- chan <- chan interface{},	
) <-chan interface{} {
	valStream := make(chan interface{})
	go func() {
		defer close(valStream)
		for {
			var stream <- chan interface{}
			select {
			case maybeStream, ok := <-chanStream:
				if ok == false {
					return
				}
				stream = maybeStream
			case <-done:
				return
			}
			for val := range orDone(done, stream) {
				select {
				case valStream <- val:
				case <-done:
				}
			}
		}
	}()
	return valStream
}

In [ ]:
%%

genVals := func() <-chan <-chan interface{} {
	chanStream := make(chan (<-chan interface{}))
	go func() {
		defer close(chanStream)
		for i := 0; i < 10; i++ {
			stream := make(chan interface{}, 1)
			stream <- i
			close(stream)
			chanStream <- stream
		}
	}()
	return chanStream
}

done := make(chan interface{})
for v := range bridge(done, genVals()) {
	fmt.Printf("%v ", v)
}

0 1 2 3 4 5 6 7 8 9 

In [ ]:
func sleep(
	done <-chan interface{},
	d time.Duration,
	zeros <-chan interface{},
) <-chan interface{} {
	zeroStream := make(chan interface{})
	go func() {
		defer close(zeroStream)
		select {
		case <-done:
			return
		case <-time.After(d):
		}
	}
	return zeroStream
}

%%
done := make(chan interface{})
defer close(done)

zeros := take(done, 3, repeat(done, 0))
short := sleep(done, 1*time.Second, zeros)
long := sleep(done, 4*time.Second, short)

pipeline := long


# gonb_ee5e6141 

 

 Cell[10]: Line 6 
 ./main.go:108:10: undefined: sleep
 
defer close(done)

zeros := take(done, repeat(done, 0), 10)
 short := sleep(done, 1*time.Second, zeros)
 long := sleep(done, 4*time.Second, short)


 

 

 Cell[10]: Line 7 
 ./main.go:109:9: undefined: sleep
 

zeros := take(done, repeat(done, 0), 10)
short := sleep(done, 1*time.Second, zeros)
 long := sleep(done, 4*time.Second, short)
 
pipeline := long

 

 

 Cell[10]: Line 9 
 ./main.go:111:1: declared and not used: pipeline
 
short := sleep(done, 1*time.Second, zeros)
long := sleep(done, 4*time.Second, short)

 pipeline := long

ERROR: failed to run "/usr/local/go/bin/go build -o /tmp/gonb_ee5e6141/gonb_ee5e6141": exit status 1

In [ ]:
// context package

var Cancelled = errors.New("context cancelled")
var DeadlineExceeded error = 
deadlineExceededError{}

type CancelFunc
type context
func Background() context
func TODO() context

func WithCancel(parent Context) (ctx Context, cancel CancelFunc)
func WithDeadline(parent Context, deadline time.Time) (Context, CancelFunc)
func WithTimeout(parent Context, timeout time.Duration) (Context, CancelFunc)

func WithValue(parent Context, key, val interface{}) Context

type Context interface {
	Deadline() (deadline time.Time, ok bool)
	Done() <-chan struct{}
	Err() error 
	Value(key interface{}) interface{}
}



In [ ]:

func printGreeting(done <-chan interface{}) error {
	greeting, err := genGreeting(done)
	if err != nil {
		return err
	}
	fmt.Printf("%s world!\n", greeting)
	return nil
}

func printFarewell(done <-chan interface{}) error {
	farewell, err := genFarewell(done)
	if err != nil {
		return err
	}
	fmt.Printf("%s world!\n", farewell)
	return nil
}

func genGreeting(done <-chan interface{}) (string, error) {
	switch locale, err := locale(done); {
	case err != nil:
		return "", err
	case locale == "EN/US":
		return "hello", nil
	}
	return "", fmt.Errorf("unsupported locale")
}

func genFarewell(done <-chan interface{}) (string, error) {
	switch locale, err := locale(done); {
	case err != nil:
		return "", err
	case locale == "EN/US":
		return "goodbye", nil
	}
	return "", fmt.Errorf("unsupported locale")
}

func locale(done <-chan interface{}) (string, error) {
	select {
	case <-done:
		return "", fmt.Errorf("cancelled")
	case <-time.After(15 * time.Second):
	}
	return "EN/US", nil
}


In [ ]:
%%

var wg sync.WaitGroup
done := make(chan interface{})
defer close(done)

wg.Add(1)
go func() {
	defer wg.Done()
	if err := printGreeting(done); err != nil {
		fmt.Printf("error: %v", err)
		return
	}
}()

wg.Add(1)
go func() {
	defer wg.Done()
	if err := printFarewell(done); err != nil {
		fmt.Printf("error: %v", err)
		return
	}
}()

wg.Wait()


goodbye world!
hello world!


In [ ]:
func printGreeting(ctx context.Context) error {
	greeting, err := genGreeting(ctx)
	if err != nil {
		return err
	}
	fmt.Printf("%s world!\n", greeting)
	return nil
}

func printFarewell(ctx context.Context) error {
	farewell, err := genFarewell(ctx)
	if err != nil {
		return err
	}
	fmt.Printf("%s world!\n", farewell)
	return nil
}

func genGreeting(ctx context.Context) (string, error) {
	ctx, cancel := context.WithTimeout(ctx, 1*time.Second)
	defer cancel()
	switch locale, err := locale(ctx); {
	case err != nil:
		return "", err
	case locale == "EN/US":
		return "hello", nil
	}
	return "", fmt.Errorf("unsupported locale")
}

func genFarewell(ctx context.Context) (string, error) {
	switch locale, err := locale(ctx); {
	case err != nil:
		return "", err
	case locale == "EN/US":
		return "goodbye", nil
	}
	return "", fmt.Errorf("unsupported locale")
}

func locale(ctx context.Context) (string, error) {
	select {
	case <-ctx.Done():
		return "", ctx.Err()
	case <-time.After(5 * time.Second):
	}
	return "EN/US", nil
}


In [ ]:
%%

var wg sync.WaitGroup
ctx, cancel := context.WithCancel(context.Background())
defer cancel()

wg.Add(1)
go func() {
	defer wg.Done()
	if err := printGreeting(ctx); err != nil {
		fmt.Printf("error: %v\n", err)
		cancel()
	}
}()

wg.Add(1)
go func() {
	defer wg.Done()
	if err := printFarewell(ctx); err != nil {
		fmt.Printf("error: %v", err)
		return
	}
}()

wg.Wait()

error: context deadline exceeded
error: context canceled

In [ ]:
func printGreeting(ctx context.Context) error {
	greeting, err := genGreeting(ctx)
	if err != nil {
		return err
	}
	fmt.Printf("%s world!\n", greeting)
	return nil
}

func printFarewell(ctx context.Context) error {
	farewell, err := genFarewell(ctx)
	if err != nil {
		return err
	}
	fmt.Printf("%s world!\n", farewell)
	return nil
}

func genGreeting(ctx context.Context) (string, error) {
	ctx, cancel := context.WithTimeout(ctx, 1*time.Second)
	defer cancel()
	switch locale, err := locale(ctx); {
	case err != nil:
		return "", err
	case locale == "EN/US":
		return "hello", nil
	}
	return "", fmt.Errorf("unsupported locale")
}

func genFarewell(ctx context.Context) (string, error) {
	switch locale, err := locale(ctx); {
	case err != nil:
		return "", err
	case locale == "EN/US":
		return "goodbye", nil
	}
	return "", fmt.Errorf("unsupported locale")
}

func locale(ctx context.Context) (string, error) {
	if deadline, ok := ctx.Deadline(); ok {
		if deadline.Sub(time.Now().Add(1*time.Second)) <= 0 {
			return "", context.DeadlineExceeded
		}
	}
	select {
	case <-ctx.Done():
		return "", ctx.Err()
	case <-time.After(5 * time.Second):
	}
	return "EN/US", nil
}

In [ ]:
%%
var wg sync.WaitGroup
ctx, cancel := context.WithCancel(context.Background())
defer cancel()

wg.Add(1)
go func() {
	defer wg.Done()
	if err := printGreeting(ctx); err != nil {
		fmt.Printf("error: %v\n", err)
		cancel()
	}
}()

wg.Add(1)
go func() {
	defer wg.Done()
	if err := printFarewell(ctx); err != nil {
		fmt.Printf("error: %v", err)
		return
	}
}()

wg.Wait()

error: context deadline exceeded
error: context canceled

In [8]:
func ProcessRequest(userID int, authToken string) error {
	ctx := context.WithValue(context.Background(), "userID", userID)
	ctx = context.WithValue(ctx, "authToken", authToken)
	return HandleResponse(ctx)
}

func HandleResponse(ctx context.Context) {
	fmt.Printf(
		"handling response for %v (%v)\n",
		ctx.Value("userID"),
		ctx.Value("authToken"),
	)
}
%%
ProcessRequest(1, "HandleResponse")

# gonb_1e41f4ed 

 

 Cell[8]: Line 4 
 ./main.go:20:9: HandleResponse(ctx) (no value) used as value
 
func ProcessRequest(userID int, authToken string) error {
	ctx := context.WithValue(context.Background(), "userID", userID)
	ctx = context.WithValue(ctx, "authToken", authToken)
 return HandleResponse(ctx)
 }

ERROR: failed to run "/usr/local/go/bin/go build -o /tmp/gonb_1e41f4ed/gonb_1e41f4ed": exit status 1

In [ ]:
func PostReport(id string) (interface{}, error) {
	result, err := lowlevel.DoWork()
	if err != nil {
		if _, ok := err.(lowlevel.Error); ok {
			err = WrapErr(err, "cannot post report with id %q", id)
		}
		return err
	}
	return result, nil
}

%%
err := PostReport("rachit")
fmt.Printf(err)

In [32]:
type MyError struct {
	Inner error
	Message string
	StackTrace string
	Misc map[string]interface{}
}

func wrapError(err error, messagef string, msgArgs ...interface{}) MyError {
	return MyError{
		Inner: err,
		Message: fmt.Sprintf(messagef, msgArgs...),
		StackTrace: string(debug.Stack()),
		Misc: make(map[string]interface{}),
	}
}

func (err MyError) Error() string {
	return err.Message
}


type LowLevelErr struct {
	error
}
func isGloballyExec(path string) (bool, error) {
	info, err := os.Stat(path)
	if err != nil {
		return false, LowLevelErr{
			(wrapError(err, err.Error())),
		}
	}
	return info.Mode().Perm()&0100 == 0100, nil
}


type IntermediateErr struct {
	error
}

func runJob(id string) error {
	const jobBinPath = "/bad/job/binary"
    isExecutable, err := isGloballyExec(jobBinPath)
    if err != nil {
        // return err 
		return IntermediateErr{
			wrapError(
				err,
				"cannot run job %q: requisite binaries not available",
				id,
			),
		}
    } else if isExecutable == false {
        return wrapError(
				nil, 
				"cannot run job %q: requisite binaries not available",
				id,
			)
    }

    return exec.Command(jobBinPath, "--id="+id).Run()
}

func handleError(key int, err error, message string) {
    log.SetPrefix(fmt.Sprintf("[logID: %v]: ", key))
    log.Printf("%#v", err) 
    fmt.Printf("[%v] %v", key, message)
}

In [33]:
%%
log.SetOutput(os.Stdout)
log.SetFlags(log.Ltime|log.LUTC)

err := runJob("1")
if err != nil {
	msg := "there was an unexpected issue: please report this is a bug."
	if _, ok := err.(IntermediateErr); ok {
		msg = err.Error()
	}
	handleError(1, err, msg)
}


[logID: 1]: 09:23:12 main.IntermediateErr{error:main.MyError{Inner:main.LowLevelErr{error:main.MyError{Inner:(*fs.PathError)(0x4000074000), Message:"stat /bad/job/binary: no such file or directory", StackTrace:"goroutine 1 [running]:\nruntime/debug.Stack()\n\t/usr/local/go/src/runtime/debug/stack.go:26 +0x64\nmain.wrapError({0x123f68, 0x4000074000}, {0x4000014150?, 0x70000000000080?}, {0x0?, 0x10?, 0x0?})\n\t/tmp/gonb_1e41f4ed/main.go:81 +0x60\nmain.isGloballyExec({0xfa85c?, 0x55084?})\n\t/tmp/gonb_1e41f4ed/main.go:39 +0x5c\nmain.runJob({0xf8cd8, 0x1})\n\t/tmp/gonb_1e41f4ed/main.go:47 +0x38\nmain.main()\n\t/tmp/gonb_1e41f4ed/main.go:92 +0xa8\n", Misc:map[string]interface {}{}}}, Message:"cannot run job \"1\": requisite binaries not available", StackTrace:"goroutine 1 [running]:\nruntime/debug.Stack()\n\t/usr/local/go/src/runtime/debug/stack.go:26 +0x64\nmain.wrapError({0x123ee8, 0x4000010050}, {0x101b86?, 0x45744?}, {0x400010ce80?, 0x0?, 0x4000046658?})\n\t/tmp/gonb_1e41f4ed/main.go:81

In [ ]:
var value interface{}
select {
case <- done:
	return
case value = <- valueStream:
}

result := reallyLongCalculation(value)
select {
case <-done:
	return
case resultStream <- result:
}

reallyLongCalculation := func(done <-chan interface{}, value interface{}) interface{} {
	intermediateResult := longCalculation(value)
	return longCalculation(intermediateResult)
}

In [ ]:
%%

doWork := func(
	done <- chan interface{},
	pulseInterval time.Duration,
) (<-chan interface{}, <-chan time.Time) {
	heartbeat := make(chan interface{})
	results := make(chan time.Time)
	go func() {
		defer close(heartbeat)
		defer close(results)

		pulse := time.Tick(pulseInterval)
		workGen := time.Tick(2 * pulseInterval)

		sendPulse := func() {
			select {
			case heartbeat <- struct{}{}:
			default:
			}
		}

		sendResult := func(r time.Time) {
			for {
				select {
				case <-done:
					return
				case <-pulse:
					sendPulse()
				case results <- r:
					return
				}
			}
		}

		for {
			select {
			case <-done:
				return
			case <-pulse:
				sendPulse()
			case r := <-workGen:
				sendResult(r)
			}
		}
	}()
	return heartbeat, results
}

done := make(chan interface{})
time.AfterFunc(5*time.Second, func() { close(done) })

const timeout = 2 * time.Second
heartbeat, results := doWork(done, timeout/2)

for {
	select {
	case _, ok := <-heartbeat:
		if ok == false {
			return
		}
		fmt.Println("❤️")
	case r, ok := <-results:
		if ok == false {
			return
		}
		fmt.Printf("results %v\n", r.Second())
	case <-time.After(timeout):
		return
	}
}

❤️
❤️
results 19
❤️
❤️
results 21
❤️


signal: interrupt


In [ ]:
%%
doWork := func(
	done <- chan interface{},
	pulseInterval time.Duration,
) (<- chan interface{}, <- chan time.Time) {
	heartbeat := make(chan interface{}, 1)
	results := make(chan time.Time)

	go func() {
		// defer close(heartbeat)
		// defer close(results)

		pulse := time.Tick(pulseInterval)
		workGen := time.Tick(2 * pulseInterval)

		sendPulse := func() {
			select {
			case heartbeat <- struct{}{}:
			default:
			}
		}

		sendResult := func(r time.Time) {
			for {
				select {
				case <-done:
					return
				case <-pulse:
					sendPulse()
				case results <- r:
					return
				}
			}
		}

		for i:= 0; i <2; i++ {
			select {
			case <-done:
				return
			case <-pulse:
				sendPulse()
			case r := <-workGen:
				sendResult(r)
			}
		}
	}()
	return heartbeat, results
}


done := make(chan interface{})
time.AfterFunc(5*time.Second, func() { close(done) })

const timeout = 2 * time.Second
heartbeat, results := doWork(done, timeout/2)

for {
	select {
	case _, ok := <-heartbeat:
		if ok == false {
			return
		}
		fmt.Println("❤️")
	case r, ok := <-results:
		if ok == false {
			return
		}
		fmt.Printf("results %v\n", r.Second())
	case <-time.After(timeout):
		fmt.Println("worker goroutine is not healthy!")
		return
	}
}

❤️
❤️
results 54
❤️
❤️
results 56
❤️
worker goroutine is not healthy!


In [52]:
%%
doWork := func(
	done <- chan interface{},
) (<- chan interface{}, <- chan int) {
	heartbeat := make(chan interface{}, 1)
	workStream := make(chan int)

	go func() {
		defer close(heartbeat)
		defer close(workStream)

		for i := 0; i < 10; i++ {
			select {
			case heartbeat <- struct{}{}:
			default:
			}
			select {
			case <-done:
				return
			case workStream <- rand.Intn(10):
			}
		}
	}()
	return heartbeat, workStream
}


done := make(chan interface{})
defer close(done)

heartbeat, results := doWork(done)

for {
	select {
	case _, ok := <-heartbeat:
		if ok == false {
			return
		}
		fmt.Println("❤️")
	case r, ok := <-results:
		if ok == false {
			return
		}
		fmt.Printf("results %v\n", r)
	}
}

❤️
results 5
❤️
results 6
❤️
results 1
❤️
results 4
❤️
results 4
❤️
results 5
❤️
results 9
❤️
results 6
❤️
results 4
❤️
results 5


In [62]:
!go test heartbeat_test.go



# command-line-arguments [command-line-arguments.test]
./heartbeat_test.go:13:16: undefined: DoWork


FAIL	command-line-arguments [build failed]
FAIL


exit status 1


In [64]:
%%
doWork := func(
	done <- chan interface{},
	id int,
	wg *sync.WaitGroup,
	result chan <- int,
) {
	started := time.Now()
	defer wg.Done()

	simulateLoadTime := time.Duration(rand.Intn(50)) * time.Millisecond

	select {
	case <-done:
	case <-time.After(simulateLoadTime):
	}

	select {
	case <-done:
	case result <- id:
	}

	took := time.Since(started)
	if took < simulateLoadTime {
		took = simulateLoadTime
	}
	fmt.Printf("%v took %v\n", id, took)
}

done := make(chan interface{})
result := make(chan int)
var wg sync.WaitGroup
for i := 0; i < 10; i++ {
	wg.Add(1)
	go doWork(done, i, &wg, result)
}

firstReturned := <-result
close(done)
wg.Wait()

fmt.Printf("Received an answer from #%v\n", firstReturned)

7 took 1.118949ms
9 took 30ms
3 took 1.34328ms
8 took 30ms
6 took 37ms
5 took 33ms
4 took 24ms
2 took 45ms
1 took 20ms
0 took 5ms
Received an answer from #7
